In [1]:
import sys
from pathlib import Path
import pandas as pd
import geopandas as gpd
sys.path.insert(0, '.')
from utils import assign_admin_level, DATA_DIR

# ── Paths ──────────────────────────────────────────────────────────────────
IATI_CSV    = DATA_DIR / 'iati-activity-locations-in-democratic-republic-of-the-congo.csv'
ADM1_SHP    = DATA_DIR / 'cod_admin_boundaries.shp/cod_admin1.shp'
ADM2_SHP    = DATA_DIR / 'cod_admin_boundaries.shp/cod_admin2.shp'
OUT_CSV     = DATA_DIR / 'iati-drc-cleaned.csv'

# ── Load data ──────────────────────────────────────────────────────────────
df        = pd.read_csv(IATI_CSV)
provinces = gpd.read_file(ADM1_SHP)
admin2    = gpd.read_file(ADM2_SHP)
print(f'Raw IATI rows: {len(df):,}')

# ── Project to metres CRS for accurate distance calculations ───────────────
METRES_CRS     = 'EPSG:32635'
provinces_proj = provinces.to_crs(METRES_CRS)
admin2_proj    = admin2.to_crs(METRES_CRS)

# ── Build projected GeoDataFrame of aid points ────────────────────────────
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['location_longitude'], df['location_latitude']),
    crs='EPSG:4326'
).to_crs(METRES_CRS)

# ── Auto-detect Admin-2 name column ───────────────────────────────────────
CANDIDATE_NAME_COLS = ['admin2Name', 'ADM2_EN', 'ADM2_FR', 'NAME_2',
                       'name', 'NAME', 'adm2_name', 'admin2name']
adm2_name_col = next((c for c in CANDIDATE_NAME_COLS if c in admin2.columns), None)
if adm2_name_col is None:
    raise ValueError(
        f'Could not auto-detect Admin-2 name column.\n'
        f'Available columns: {admin2.columns.tolist()}'
    )
print(f'Admin-1 name column: adm1_name')
print(f'Admin-2 name column: {adm2_name_col}')

# ── Admin-1 province assignment (point-in-polygon + nearest fallback) ─────
df['province_name'] = assign_admin_level(
    gdf, provinces_proj[['adm1_name', 'geometry']].rename(columns={'adm1_name': '_name'}),
    name_col='_name'
)

# ── Admin-2 territory assignment (same logic, different boundary layer) ────
# Rebuild GeoDataFrame from province-matched rows only
gdf2 = gpd.GeoDataFrame(
    df.dropna(subset=['province_name']),
    geometry=gpd.points_from_xy(
        df.dropna(subset=['province_name'])['location_longitude'],
        df.dropna(subset=['province_name'])['location_latitude']
    ),
    crs='EPSG:4326'
).to_crs(METRES_CRS)

df.loc[gdf2.index, 'admin2_name'] = assign_admin_level(
    gdf2, admin2_proj[[adm2_name_col, 'geometry']].rename(columns={adm2_name_col: '_name'}),
    name_col='_name'
)

# Drop rows still unmatched at province level (outside DRC)
df_final = df.dropna(subset=['province_name'])

# ══════════════════════════════════════════════════════════════════════════
# Deduplicate
# ══════════════════════════════════════════════════════════════════════════
print(f'\nBefore dedup: {len(df_final):,} rows')
df_final = df_final.drop_duplicates()
print(f'After dropping exact dupes: {len(df_final):,} rows')

# Some projects appear at multiple coordinates in the same province;
# deduplicate on key fields so each project-province pair is counted once.
key_cols = ['aid', 'province_name', 'day_start', 'day_end', 'description', 'spend']
df_final = df_final.drop_duplicates(subset=key_cols)
print(f'After dropping key-col dupes: {len(df_final):,} rows')

print(f'\nUnique aid projects: {df_final["aid"].nunique()}')
print(f'NAs in province_name: {df_final["province_name"].isna().sum()}')
print(f'NAs in admin2_name  : {df_final["admin2_name"].isna().sum()}')
print(f'Unique provinces    : {df_final["province_name"].nunique()}')
print(f'Unique admin2 towns : {df_final["admin2_name"].nunique()}')
print(f'\nTop provinces:\n{df_final["province_name"].value_counts().head(10).to_string()}')

# ── Save ───────────────────────────────────────────────────────────────────
df_final.to_csv(OUT_CSV, index=False)
print(f'\nSaved → {OUT_CSV}')


Raw IATI rows: 34,596
Admin-1 name column: adm1_name
Admin-2 name column: adm2_name



Before dedup: 28,324 rows
After dropping exact dupes: 6,332 rows
After dropping key-col dupes: 4,917 rows

Unique aid projects: 2836
NAs in province_name: 0
NAs in admin2_name  : 0
Unique provinces    : 26
Unique admin2 towns : 148

Top provinces:
province_name
Kinshasa          1338
Nord-Kivu          445
Kasaï              434
Sud-Kivu           394
Sankuru            249
Ituri              229
Tanganyika         198
Haut-Katanga       174
Tshopo             173
Kasaï-Oriental     145



Saved → /Users/jackzipper/QSS20/final_project/final_project_data/iati-drc-cleaned.csv
